### Setup

Install dependencies

In [1]:
# # PyTorch
!uv pip install torch torchvision torchaudio -q

# # HuggingFace ecosystem
!uv pip install transformers[torch] datasets -q

# # Utilities
!uv pip install ipywidgets tqdm pyarrow scikit-learn tensorboardX -q

Set huggingface access-token for loading pre-trained models

In [2]:
import os
from huggingface_hub import login
from dotenv import load_dotenv

token = None # "YOUR_TOKEN"

load_dotenv()

hf_token = token or os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Supress unwanted warnings

In [3]:
import warnings

warnings.filterwarnings("ignore")

### Initialize fine-tuning

Import packages

In [4]:
import notebook_init  # Ensure max 2 GPUs used
import json
import numpy as np
import os
import torch
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import ParameterGrid, StratifiedKFold
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DefaultDataCollator,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer
)
from datasets import Dataset

LABEL_MAP = {
    "HUMAN_GENERATED": 0,
    "MACHINE_GENERATED": 1,
}

[env_init] Using GPU: 0,1


Tokenize and chunk

In [5]:
def chunk_code(tokenizer, code, window=512, stride=0):
    """
    Chunk a single code string into fixed-length token windows.
    Ensures that the FULL code snippet is captured and all chunks are exactly 'window' length.
    """
    # Tokenize the full code without truncation
    full_encoded = tokenizer(
        code,
        add_special_tokens=True,
        truncation=False,
    )
    
    input_ids = full_encoded["input_ids"]
    attention_mask = full_encoded["attention_mask"]
    
    # Split into chunks of size 'window'
    chunks = []
    
    # If stride is 0, we just do a simple split
    step = window if stride <= 0 else (window - stride)
    
    for i in range(0, len(input_ids), step):
        chunk_ids = input_ids[i : i + window]
        chunk_mask = attention_mask[i : i + window]
        
        # Handle the final chunk padding
        if len(chunk_ids) < window:
            padding_length = window - len(chunk_ids)
            # Use tokenizer.pad_token_id explicitly
            chunk_ids = chunk_ids + [tokenizer.pad_token_id] * padding_length
            chunk_mask = chunk_mask + [0] * padding_length
            
        chunks.append({
            "input_ids": chunk_ids,
            "attention_mask": chunk_mask
        })
        
        # Break if we've reached the end of the ids
        if i + window >= len(input_ids):
            break

    return chunks


def map_chunks(batch, tokenizer, window):
    """Map a batch of code/label pairs into tokenized chunk features."""
    all_input_ids, all_attention_mask, all_labels = [], [], []

    for code, label in zip(batch["code"], batch["label"]):
        # Normalize label
        if isinstance(label, str):
            label_key = label.strip().upper()
            if label_key not in LABEL_MAP:
                raise ValueError(f"Unknown label: {label}")
            label_id = LABEL_MAP[label_key]
        else:
            label_id = int(label)

        # Process code into chunks
        encoded_chunks = chunk_code(tokenizer, code, window=window)
        for enc in encoded_chunks:
            all_input_ids.append(enc["input_ids"])
            all_attention_mask.append(enc["attention_mask"])
            all_labels.append(label_id)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "labels": all_labels,
    }


def tokenize_and_chunk(dataset, tokenizer, window, num_proc=1):
    """Tokenize and chunk an entire dataset into model-ready features."""
    return dataset.map(
        map_chunks,
        batched=True,
        remove_columns=dataset.column_names,
        fn_kwargs={"tokenizer": tokenizer, "window": window},
        num_proc=num_proc
    )


def load_tokenizer(model_name):
    """Load a tokenizer and ensure a usable pad token is set and configured."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})
            
    # Explicitly ensure the model config will see this pad token
    return tokenizer

def load_classifier_model(model_name, tokenizer):
    """Load model and ensure pad_token_id is correctly set in the config."""
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        pad_token_id=tokenizer.pad_token_id  # Pass it here explicitly
    )
    
    # Validation/Sanity Check for DeepSeek-style NaN initialization
    # If the score weights are NaN (common in some Llama-based classification loads)
    if torch.isnan(model.score.weight).any():
        print("Fixing NaN weights in classification head...")
        with torch.no_grad():
            model.score.weight.fill_(0.0)
            
    return model

Set dataset and config paths

In [6]:
current_dir = Path.cwd()

if current_dir.name == "notebooks" and (
    Path.exists(current_dir.parent / Path("configs/fine_tune"))
    and Path.exists(current_dir.parent / Path("data"))
):
    os.chdir(current_dir.parent)

print(f"Current directory: {Path.cwd().relative_to(Path.home())}")

cfg_path = "configs/fine_tune/test.json"
ds_path = "data/final_samples/test_samples.parquet"

Current directory: projects/dt133g-thesis-project


### Run grid search + CV fold

In [ ]:
BEST_METRIC = "f1"
EARLY_STOP_PATIENCE = 3

def compute_metrics(eval_pred):
    """Compute accuracy, precision, recall, and F1 from model logits."""
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def run_cv(dataset, tokenizer, model_name, folds, lr, batch_size, grad_acc_steps, epochs, output_dir, window=512):
    """Run stratified K-fold training and return per-fold metrics."""
    labels = dataset["label"]
    skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    fold_metrics = []
    for fold, (train_idx, test_idx) in enumerate(
        skf.split(np.arange(len(labels)), labels), start=1
    ):
        print(f"\n── Start fold {fold}/{folds} ──")

        train_ds = dataset.select(train_idx)
        test_ds = dataset.select(test_idx)
        print("Train & Test data selected\n")

        print("Tokenizing train data into chunks...")
        train_tok = tokenize_and_chunk(train_ds, tokenizer, window)
        print("Tokenizing test data into chunks...")
        test_tok = tokenize_and_chunk(test_ds, tokenizer, window)

        print("\nLoading model for fold...")
        model = load_classifier_model(model_name, tokenizer)

        out_dir = Path(output_dir) / f"fold_{fold}"
        os.environ["TENSORBOARD_LOGGING_DIR"] = str(out_dir)

        args = TrainingArguments(
            output_dir=str(out_dir),
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            gradient_accumulation_steps=grad_acc_steps,
            learning_rate=lr,
            num_train_epochs=epochs,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=2,
            logging_steps=20,
            report_to=["tensorboard"],
            load_best_model_at_end=True,
            metric_for_best_model=BEST_METRIC,
            bf16=True
        )

        trainer = Trainer(
            model=model,
            args=args,
            data_collator=DefaultDataCollator(),
            train_dataset=train_tok,
            eval_dataset=test_tok,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)]
        )

        print("\n===========================")
        print("Training Arguments:")
        print(f" Effective batch size: {batch_size * grad_acc_steps}")
        print(f" Learning rate: {lr}")
        print(f" Epochs: {epochs}")
        print("===========================")

        print("\n  Training...")
        trainer.train()
        print("\n  Evaluating...")
        metrics = trainer.evaluate()
        fold_metrics.append(metrics)

    metric_keys = fold_metrics[0].keys()
    mean_metrics = {k: float(np.mean([m[k] for m in fold_metrics])) for k in metric_keys}
    std_metrics = {k: float(np.std([m[k] for m in fold_metrics])) for k in metric_keys}

    print(f"\n── All folds complete ──")

    return {
        "folds": fold_metrics,
        "mean": mean_metrics,
        "std": std_metrics,
    }

torch.cuda.empty_cache()

def run_grid_search(config_path, dataset_path):
    """Run a hyperparameter grid search with stratified CV."""
    cfg = json.load(open(config_path))
    model_name = cfg["model_name"]
    grid = ParameterGrid(cfg["params"])

    results_dir = Path("output/fine_tune/folds") / Path(model_name)
    results_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = load_tokenizer(model_name)
    dataset = Dataset.from_parquet(dataset_path, columns=["code", "label"])

    for run, params in enumerate(grid, start=1):
        lr = params["learning_rate"]
        bs = params["batch_size"]
        gacc = params["grad_acc_steps"]
        run_name = f"lr{lr}_bs{bs}"
        out_dir = results_dir / run_name
        out_dir.mkdir(exist_ok=True)

        log_dir = out_dir / Path("tensorboard")
        log_dir.mkdir(parents=True, exist_ok=True)
        os.environ["TENSORBOARD_LOGGING_DIR"] = str(log_dir)

        print("Running grid search")
        print(f"     (run {run}/{len(grid)})")
        print("────────────────────")

        metrics = run_cv(
            dataset,
            tokenizer=tokenizer,
            model_name=model_name,
            folds=cfg["cv_folds"],
            lr=lr,
            batch_size=bs,
            grad_acc_steps=gacc,
            epochs=params["epochs"],
            output_dir=str(out_dir),
            window=cfg.get("window", 512)
        )

        with open(out_dir / "metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)

run_grid_search(config_path=cfg_path, dataset_path=ds_path)

Running grid search
     (run 1/1)
────────────────────

── Start fold 1/3 ──
Train & Test data selected

Tokenizing train data into chunks...
Tokenizing test data into chunks...

Loading model for fold...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: deepseek-ai/deepseek-coder-1.3b-base
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fixing NaN weights in classification head...

Training Arguments:
 Effective batch size: 32
 Optimization steps: 328
 Learning rate: 5e-05
 Epochs: 1

  Training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.568701,0.341182,0.950342,0.952467,0.948912,0.950052


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.631323,0.368910,0.943507,0.944279,0.942787,0.943325


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


  Evaluating...


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.631323,0.368910,1,0.943507,0.944279,0.942787,0.943325



── All folds complete ──


### Train best model

In [8]:
torch.cuda.empty_cache()

def train_best_model(config_path, dataset_path, metric_name="f1"):
    """Train a final model on the full dataset using best CV settings.

    Args:
        config_path: Path to the JSON config used for the grid search.
        dataset_path: Path to a parquet dataset with "code" and "label" columns.
        metric_name: Metric key used to select the best run (default: "f1").
    """
    cfg = json.load(open(config_path))
    model_name = cfg["model_name"]
    window = cfg.get("window", 512)

    tokenizer = load_tokenizer(model_name)

    results_dir = Path("output/fine_tune/models") / Path(model_name)
    results_dir.mkdir(parents=True, exist_ok=True)
    
    log_dir = results_dir / Path("tensorboard")
    log_dir.mkdir(parents=True, exist_ok=True)
    os.environ["TENSORBOARD_LOGGING_DIR"] = str(log_dir)
    
    print("── Retrieving best hyperparameters...")

    best = None
    for run_dir in sorted(Path(f"output/fine_tune/folds/{model_name}").glob("lr*_bs*")):
        metrics_path = run_dir / "metrics.json"
        if not metrics_path.exists():
            continue
        metrics = json.load(open(metrics_path))
        score = metrics.get("mean", {}).get(f"eval_{metric_name}")
        if score is None:
            continue

        if best is None or score > best["score"]:
            lr_str, bs_str = run_dir.name.replace("lr", "").split("_bs")
            best = {
                "score": score,
                "lr": float(lr_str),
                "batch_size": int(bs_str),
            }

    if best is None:
        raise ValueError("No metrics found to select the best run.")

    dataset = Dataset.from_parquet(dataset_path, columns=["code", "label"])

    print("\nTokenizing full dataset into chunks...")
    train_tok = tokenize_and_chunk(dataset, tokenizer, window)

    print("\nLoading model for final training...")
    model = load_classifier_model(model_name, tokenizer)

    args = TrainingArguments(
        output_dir=str(results_dir),
        per_device_train_batch_size=best["batch_size"],
        per_device_eval_batch_size=best["batch_size"],
        gradient_accumulation_steps=cfg["params"]["grad_acc_steps"][0],
        learning_rate=best["lr"],
        num_train_epochs=cfg["params"]["epochs"][0],
        save_strategy="epoch",
        save_total_limit=2,
        logging_steps=20,
        report_to=["tensorboard"],
        bf16=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok
    )

    print("\nTraining final model...")
    trainer.train()
    trainer.save_model(str(results_dir))

    with open(results_dir / "best_run.json", "w") as f:
        json.dump(best, f, indent=2)
    
    print(f"\n── Model saved to {str(results_dir)}")

train_best_model(config_path=cfg_path, dataset_path=ds_path)


── Retrieving best hyperparameters...

Tokenizing full dataset into chunks...

Loading model for final training...


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

[transformers] LlamaForSequenceClassification LOAD REPORT from: deepseek-ai/deepseek-coder-1.3b-base
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fixing NaN weights in classification head...

Training final model...


Step,Training Loss
20,2.459570
40,1.485352
60,1.098145
80,0.956396
100,1.022070
120,0.731787
140,0.710010
160,0.683643
180,0.613354
200,0.720898


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


── Model saved to output/fine_tune/models/deepseek-ai/deepseek-coder-1.3b-base
